# 03 - Project Momentum Scoring and ML Similarity Layer

This notebook reads the standardized public urban renewal dataset and adds transparent scoring, cautious ML-based similarity, and dashboard-ready explanatory columns.

The main dashboard should rely primarily on the rule-based **Project Momentum Score**. The ML layer is a secondary **Indicative Advancement Score** based on similarity to publicly advanced projects using non-status public features.

**Disclaimer:** The scores are indicative only. They are based on public data and may be incomplete or outdated. They are not legal advice, planning advice, real-estate advice, real-estate prediction, or a binding assessment.

## 1 - Imports and Paths

Import core libraries, optional scikit-learn components, define project paths, and create required folders.

In [1]:
from __future__ import annotations

import json
import re
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import joblib
    from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
    from sklearn.base import clone
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        roc_auc_score,
        classification_report,
        confusion_matrix,
    )
    SKLEARN_AVAILABLE = True
except Exception as exc:
    SKLEARN_AVAILABLE = False
    SKLEARN_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"

warnings.filterwarnings("default")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR = PROJECT_ROOT / "models"

INPUT_STANDARDIZED_PATH = PROCESSED_DIR / "urban_renewal_standardized.csv"
OUTPUT_SCORED_PATH = PROCESSED_DIR / "urban_renewal_scored.csv"
CITY_SCORED_SUMMARY_PATH = PROCESSED_DIR / "urban_renewal_city_scored_summary.csv"
STATUS_SCORED_SUMMARY_PATH = PROCESSED_DIR / "urban_renewal_scored_status_summary.csv"
SCORING_REPORT_PATH = METADATA_DIR / "scoring_report.csv"
ML_MODEL_METRICS_PATH = METADATA_DIR / "ml_model_metrics.csv"
ML_FEATURE_IMPORTANCE_PATH = METADATA_DIR / "ml_feature_importance.csv"
SCORED_DATA_DICTIONARY_PATH = METADATA_DIR / "urban_renewal_scored_data_dictionary.csv"
ML_SIMILARITY_MODEL_PATH = MODELS_DIR / "ml_similarity_pipeline.joblib"
ML_SIMILARITY_METADATA_PATH = MODELS_DIR / "ml_similarity_metadata.json"

for folder in [DATA_DIR, PROCESSED_DIR, METADATA_DIR, OUTPUTS_DIR, MODELS_DIR, PROJECT_ROOT / "notebooks"]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input standardized path: {INPUT_STANDARDIZED_PATH.relative_to(PROJECT_ROOT)}")
print(f"sklearn available: {SKLEARN_AVAILABLE}")
if not SKLEARN_AVAILABLE:
    print(f"sklearn import issue: {SKLEARN_IMPORT_ERROR}")

Project root: C:\Users\Guy\Desktop\Birthday present
Input standardized path: data\processed\urban_renewal_standardized.csv
sklearn available: True


## 2 - Load Standardized Dataset

Load the Notebook 02 standardized dataset. Stop with a clear error if it is missing.

In [2]:
if not INPUT_STANDARDIZED_PATH.exists():
    raise FileNotFoundError("Notebook 02 output not found. Run Notebook 02 first.")

standardized_df = pd.read_csv(INPUT_STANDARDIZED_PATH, encoding="utf-8-sig")

print(f"Standardized shape: {standardized_df.shape[0]:,} rows x {standardized_df.shape[1]:,} columns")
print("Columns:")
print(list(standardized_df.columns))
display(standardized_df.head())

for col in ["planning_status_normalized", "confidence_level", "data_quality_flag"]:
    if col in standardized_df.columns:
        print(f"\n{col} counts:")
        display(standardized_df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="num_records"))
    else:
        print(f"\n{col} is missing from standardized_df")

Standardized shape: 938 rows x 34 columns
Columns:
['record_id', 'source_record_id', 'city', 'city_code', 'neighborhood', 'street_or_area', 'complex_name', 'plan_number', 'renewal_type', 'planning_status_raw', 'planning_status_normalized', 'declared_complex', 'existing_units', 'additional_units', 'proposed_units', 'permits_total', 'declaration_date', 'validity_year', 'in_execution', 'mavat_url', 'map_url', 'source_name', 'source_url', 'source_type', 'original_resource_id', 'last_updated', 'confidence_level', 'data_confidence_score', 'data_quality_flag', 'lawyer_note', 'original_file_path', 'original_row_index', 'ingestion_timestamp', 'detected_relevance_reason']


,record_id,source_record_id,city,city_code,neighborhood,street_or_area,complex_name,plan_number,renewal_type,planning_status_raw,...,original_resource_id,last_updated,confidence_level,data_confidence_score,data_quality_flag,lawyer_note,original_file_path,original_row_index,ingestion_timestamp,detected_relevance_reason
0,UR_4001,4001,גבעתים,6300,NaN,NaN,ערבי נחל,גב/490,מיסוי,תכנית מאושרת - אחרי רישוי,...,f65a0daf-f737-49c5-9424-d378d52104f5,2026-05-26,HIGH,100,OK,Official record found with indication of post-...,data\manual_sources\data_gov_f65a0daf_records....,0,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
1,UR_4005,4005,קרית אונו,2620,NaN,NaN,ישעיהו,תממ/284,מיסוי,תכנית מאושרת במימוש,...,f65a0daf-f737-49c5-9424-d378d52104f5,2026-05-26,HIGH,100,OK,Official record indicates implementation/const...,data\manual_sources\data_gov_f65a0daf_records....,1,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
2,UR_4006,4006,קרית אונו,2620,NaN,NaN,שאול המלך,קא/מק/61/285/א,מיסוי,תכנית מאושרת - אחרי רישוי,...,f65a0daf-f737-49c5-9424-d378d52104f5,2026-05-26,HIGH,100,OK,Official record found with indication of post-...,data\manual_sources\data_gov_f65a0daf_records....,2,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
3,UR_4010,4010,ראשון לציון,8300,NaN,NaN,רמת אליהו (פוזננסקי),413-0292680,מיסוי,תכנית מאושרת במימוש,...,f65a0daf-f737-49c5-9424-d378d52104f5,2026-05-26,HIGH,100,OK,Official record indicates implementation/const...,data\manual_sources\data_gov_f65a0daf_records....,3,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
4,UR_4011,4011,ראשון לציון,8300,NaN,NaN,סלע,רצ/מק/1/13/19/4,מיסוי,תכנית מאושרת במימוש,...,f65a0daf-f737-49c5-9424-d378d52104f5,2026-05-26,HIGH,100,OK,Official record indicates implementation/const...,data\manual_sources\data_gov_f65a0daf_records....,4,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...



planning_status_normalized counts:


,planning_status_normalized,num_records
0,PLAN_IN_PROGRESS,385
1,PLAN_APPROVED,325
2,PERMIT_APPROVED,173
3,CONSTRUCTION,54
4,UNKNOWN,1



confidence_level counts:


,confidence_level,num_records
0,HIGH,936
1,MEDIUM,2



data_quality_flag counts:


,data_quality_flag,num_records
0,OK,936
1,NEGATIVE_UNIT_VALUE,1
2,INVALID_NUMERIC_UNITS|COLUMN_SHIFT_OR_SOURCE_A...,1


## 3 - Helper Functions

Reusable utilities for robust feature engineering, score labels, quality penalties, explanations, and dashboard notes.

In [3]:
def safe_col(df: pd.DataFrame, col: str, default: Any = pd.NA) -> pd.Series:
    if col in df.columns:
        return df[col]
    return pd.Series(default, index=df.index, dtype="object")


def to_bool_indicator(series: pd.Series) -> pd.Series:
    cleaned = series.copy()
    if cleaned.dtype == "bool":
        return cleaned.fillna(False).astype(bool)
    text = cleaned.astype("string").str.strip()
    empty_like = text.isna() | text.str.lower().isin(["", "nan", "none", "null", "-", "--"])
    return (~empty_like).fillna(False).astype(bool)


def safe_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def safe_date(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce")


def clamp_score(x: Any, lower: float = 0, upper: float = 100) -> Any:
    if pd.isna(x):
        return pd.NA
    return max(lower, min(upper, float(x)))


def label_score(score: Any) -> str:
    if pd.isna(score):
        return "UNKNOWN"
    score = float(score)
    if score >= 81:
        return "VERY_HIGH"
    if score >= 61:
        return "HIGH"
    if score >= 31:
        return "MODERATE"
    return "LOW"


QUALITY_PENALTIES = {
    "MISSING_STATUS": 10,
    "MISSING_LINKS": 10,
    "MISSING_CITY": 15,
    "MISSING_COMPLEX_NAME": 15,
    "INVALID_NUMERIC_UNITS": 20,
    "NEGATIVE_UNIT_VALUE": 20,
    "UNIT_LOGIC_ANOMALY": 20,
    "COLUMN_SHIFT_OR_SOURCE_ANOMALY": 30,
    "DUPLICATE_RECORD_ID": 25,
}


def quality_penalty_from_flag(flag: Any) -> int:
    if pd.isna(flag) or str(flag).strip() == "" or str(flag).strip() == "OK":
        return 0
    parts = [part.strip() for part in str(flag).split("|") if part.strip()]
    penalty = sum(QUALITY_PENALTIES.get(part, 0) for part in parts)
    return int(min(50, penalty))


def has_text_value(value: Any) -> bool:
    if pd.isna(value):
        return False
    text = str(value).strip()
    return text != "" and text.lower() not in {"nan", "none", "null", "-", "--"}


def official_public_source(row: pd.Series) -> bool:
    blob = " ".join(str(row.get(col, "")) for col in ["source_name", "source_url", "source_type", "original_resource_id"]).lower()
    return any(term in blob for term in ["data.gov", "gov", "government", "official", "ממשל", "משרד", "רשות"])


def create_project_momentum_explanation(row: pd.Series) -> str:
    label = row.get("project_momentum_label", "UNKNOWN")
    status = row.get("planning_status_normalized", "UNKNOWN")
    parts: List[str] = []

    if status in {"PERMIT_APPROVED", "CONSTRUCTION", "COMPLETED"}:
        parts.append("Advanced public status with strong planning-maturity signal.")
    elif status == "PLAN_APPROVED":
        parts.append("High maturity score due to approved planning status.")
    elif status == "PLAN_IN_PROGRESS":
        parts.append("Moderate score: official record exists, but planning appears to still be in progress.")
    elif status == "UNKNOWN":
        parts.append("Limited score because the public planning status is unclear.")
    else:
        parts.append("Score reflects available public planning status and source traceability.")

    if bool(row.get("has_mavat_url", False)):
        parts.append("Mavat link available.")
    if row.get("data_quality_flag") != "OK":
        parts.append("Score reduced due to data-quality issues.")
    if label in {"VERY_HIGH", "HIGH"}:
        parts.append("Manual verification still required.")
    return " ".join(parts)


def create_dashboard_note(row: pd.Series) -> str:
    label = row.get("project_momentum_label", "UNKNOWN")
    if label == "VERY_HIGH":
        note = "Public data indicates an advanced or highly mature planning stage. Manual verification is still required."
    elif label == "HIGH":
        note = "Public data indicates meaningful planning advancement. Recommended to inspect official documents."
    elif label == "MODERATE":
        note = "Official record exists, but the public status suggests the project may still be progressing."
    elif label == "LOW":
        note = "Limited or early-stage public indication based on available data."
    else:
        note = "Public-data maturity signal is unclear; manual review is recommended."
    if row.get("data_quality_flag") != "OK":
        note += " Data-quality warning exists."
    if bool(row.get("has_mavat_url", False)):
        note += " Mavat link available."
    return note


def ml_similarity_label(score: Any) -> str:
    if pd.isna(score):
        return "ML_NOT_AVAILABLE"
    score = float(score)
    if score >= 70:
        return "HIGH_SIMILARITY"
    if score >= 40:
        return "MEDIUM_SIMILARITY"
    return "LOW_SIMILARITY"

## 4 - Feature Engineering

Create dashboard-friendly indicators, date features, and cautious unit ratios without altering original standardized values.

In [4]:
scored_df = standardized_df.copy()

OPTIONAL_COLUMNS = [
    "record_id",
    "source_record_id",
    "city",
    "city_code",
    "complex_name",
    "plan_number",
    "renewal_type",
    "planning_status_raw",
    "planning_status_normalized",
    "declared_complex",
    "existing_units",
    "additional_units",
    "proposed_units",
    "permits_total",
    "declaration_date",
    "validity_year",
    "in_execution",
    "mavat_url",
    "map_url",
    "source_name",
    "source_url",
    "source_type",
    "original_resource_id",
    "last_updated",
    "confidence_level",
    "data_confidence_score",
    "data_quality_flag",
    "lawyer_note",
]

for col in OPTIONAL_COLUMNS:
    if col not in scored_df.columns:
        scored_df[col] = pd.NA

scored_df["has_plan_number"] = to_bool_indicator(scored_df["plan_number"])
scored_df["has_mavat_url"] = to_bool_indicator(scored_df["mavat_url"])
scored_df["has_map_url"] = to_bool_indicator(scored_df["map_url"])

for col in ["existing_units", "additional_units", "proposed_units", "permits_total", "validity_year", "data_confidence_score"]:
    scored_df[col] = safe_numeric(scored_df[col])

scored_df["has_existing_units"] = scored_df["existing_units"].notna() & (scored_df["existing_units"] > 0)
scored_df["has_proposed_units"] = scored_df["proposed_units"].notna() & (scored_df["proposed_units"] > 0)
scored_df["has_permits"] = scored_df["permits_total"].notna() & (scored_df["permits_total"] > 0)
scored_df["has_quality_issue"] = scored_df["data_quality_flag"].fillna("OK").astype(str) != "OK"

scored_df["declaration_date_parsed"] = safe_date(scored_df["declaration_date"])
scored_df["declaration_year"] = scored_df["declaration_date_parsed"].dt.year.astype("Int64")
current_year = datetime.now().year
scored_df["years_since_declaration"] = (current_year - scored_df["declaration_year"]).astype("Int64")
scored_df.loc[scored_df["declaration_year"].isna(), "years_since_declaration"] = pd.NA

quality_text = scored_df["data_quality_flag"].fillna("OK").astype(str)
ratio_blocked = quality_text.str.contains("NEGATIVE_UNIT_VALUE|COLUMN_SHIFT_OR_SOURCE_ANOMALY", regex=True, na=False)
valid_ratio_base = scored_df["existing_units"].notna() & (scored_df["existing_units"] > 0) & (~ratio_blocked)

scored_df["proposed_to_existing_ratio"] = pd.NA
ratio_mask = valid_ratio_base & scored_df["proposed_units"].notna() & (scored_df["proposed_units"] >= 0)
scored_df.loc[ratio_mask, "proposed_to_existing_ratio"] = scored_df.loc[ratio_mask, "proposed_units"] / scored_df.loc[ratio_mask, "existing_units"]

scored_df["additional_to_existing_ratio"] = pd.NA
ratio_mask = valid_ratio_base & scored_df["additional_units"].notna() & (scored_df["additional_units"] >= 0)
scored_df.loc[ratio_mask, "additional_to_existing_ratio"] = scored_df.loc[ratio_mask, "additional_units"] / scored_df.loc[ratio_mask, "existing_units"]

print("Feature engineering preview:")
display(scored_df[["record_id", "city", "has_plan_number", "has_mavat_url", "declaration_year", "years_since_declaration", "proposed_to_existing_ratio", "additional_to_existing_ratio"]].head())

Feature engineering preview:


,record_id,city,has_plan_number,has_mavat_url,declaration_year,years_since_declaration,proposed_to_existing_ratio,additional_to_existing_ratio
0,UR_4001,גבעתים,True,True,2006,20,4.206349,0.857143
1,UR_4005,קרית אונו,True,True,2006,20,2.0,1.0
2,UR_4006,קרית אונו,True,True,2004,22,2.95,0.266667
3,UR_4010,ראשון לציון,True,True,2017,9,5.0,4.0
4,UR_4011,ראשון לציון,True,True,2012,14,4.897527,0.0


## 5 - Rule-Based Planning Maturity Score

Create the main public-data maturity component from normalized planning status.

In [5]:
PLANNING_MATURITY_MAP = {
    "UNKNOWN": 0,
    "POLICY_AREA_ONLY": 10,
    "DECLARED_COMPLEX": 20,
    "PLAN_IN_PROGRESS": 40,
    "PLAN_DEPOSITED": 55,
    "PLAN_APPROVED": 70,
    "PERMIT_REQUESTED": 80,
    "PERMIT_APPROVED": 90,
    "CONSTRUCTION": 95,
    "COMPLETED": 100,
}

scored_df["planning_maturity_score"] = (
    scored_df["planning_status_normalized"].fillna("UNKNOWN").map(PLANNING_MATURITY_MAP).fillna(0).astype(float)
)

display(scored_df[["planning_status_normalized", "planning_maturity_score"]].drop_duplicates().sort_values("planning_maturity_score"))

,planning_status_normalized,planning_maturity_score
549,UNKNOWN,0.0
34,PLAN_IN_PROGRESS,40.0
8,PLAN_APPROVED,70.0
0,PERMIT_APPROVED,90.0
1,CONSTRUCTION,95.0


## 6 - Source Strength Score

Score how strongly each record is supported by traceable public-source evidence.

In [6]:
def compute_source_strength(row: pd.Series) -> float:
    score = 0
    if official_public_source(row):
        score += 25
    if bool(row.get("has_plan_number", False)):
        score += 25
    if bool(row.get("has_mavat_url", False)):
        score += 25
    if bool(row.get("has_map_url", False)):
        score += 15
    if has_text_value(row.get("original_resource_id")):
        score += 10
    return float(clamp_score(score))

scored_df["source_strength_score"] = scored_df.apply(compute_source_strength, axis=1)

display(scored_df[["record_id", "source_strength_score", "has_plan_number", "has_mavat_url", "has_map_url", "original_resource_id"]].head())

,record_id,source_strength_score,has_plan_number,has_mavat_url,has_map_url,original_resource_id
0,UR_4001,100.0,True,True,True,f65a0daf-f737-49c5-9424-d378d52104f5
1,UR_4005,100.0,True,True,True,f65a0daf-f737-49c5-9424-d378d52104f5
2,UR_4006,100.0,True,True,True,f65a0daf-f737-49c5-9424-d378d52104f5
3,UR_4010,100.0,True,True,True,f65a0daf-f737-49c5-9424-d378d52104f5
4,UR_4011,100.0,True,True,True,f65a0daf-f737-49c5-9424-d378d52104f5


## 7 - Scale Score

Create a cautious project-scale signal. This is not a progress score.

In [7]:
def compute_scale_score(row: pd.Series) -> float:
    flag = str(row.get("data_quality_flag") or "OK")
    if "NEGATIVE_UNIT_VALUE" in flag or "COLUMN_SHIFT_OR_SOURCE_ANOMALY" in flag:
        return 30.0

    proposed_units = row.get("proposed_units")
    if pd.isna(proposed_units):
        score = 30
    elif proposed_units < 100:
        score = 35
    elif proposed_units < 300:
        score = 50
    elif proposed_units < 700:
        score = 65
    elif proposed_units < 1500:
        score = 80
    else:
        score = 90

    ratio = row.get("proposed_to_existing_ratio")
    if not pd.isna(ratio):
        if ratio >= 2:
            score += 10
        elif ratio >= 1.5:
            score += 5
    return float(clamp_score(score))

scored_df["scale_score"] = scored_df.apply(compute_scale_score, axis=1)

display(scored_df[["record_id", "proposed_units", "existing_units", "proposed_to_existing_ratio", "data_quality_flag", "scale_score"]].head())

,record_id,proposed_units,existing_units,proposed_to_existing_ratio,data_quality_flag,scale_score
0,UR_4001,530.0,126.0,4.206349,OK,75.0
1,UR_4005,396.0,198.0,2.0,OK,75.0
2,UR_4006,531.0,180.0,2.95,OK,75.0
3,UR_4010,290.0,58.0,5.0,OK,60.0
4,UR_4011,1386.0,283.0,4.897527,OK,90.0


## 8 - Data Quality Penalty

Convert data quality flags into a capped penalty used by the Project Momentum Score.

In [8]:
scored_df["data_quality_penalty"] = scored_df["data_quality_flag"].apply(quality_penalty_from_flag)

display(scored_df["data_quality_penalty"].value_counts(dropna=False).rename_axis("data_quality_penalty").reset_index(name="num_records"))

,data_quality_penalty,num_records
0,0,936
1,20,1
2,50,1


## 9 - Project Momentum Score

Combine planning maturity, source strength, data confidence, scale, and data-quality penalties into the primary explainable score.

In [9]:
formula_scale = scored_df["scale_score"].fillna(50)
formula_confidence = scored_df["data_confidence_score"].fillna(50)

raw_momentum = (
    0.55 * scored_df["planning_maturity_score"].fillna(0)
    + 0.20 * scored_df["source_strength_score"].fillna(0)
    + 0.15 * formula_confidence
    + 0.10 * formula_scale
    - scored_df["data_quality_penalty"].fillna(0)
)

scored_df["project_momentum_score"] = raw_momentum.apply(lambda x: round(float(clamp_score(x)), 1))
scored_df["project_momentum_label"] = scored_df["project_momentum_score"].apply(label_score)
scored_df["project_momentum_explanation"] = scored_df.apply(create_project_momentum_explanation, axis=1)

print("Project Momentum Score label counts:")
display(scored_df["project_momentum_label"].value_counts(dropna=False).rename_axis("project_momentum_label").reset_index(name="num_records"))
display(scored_df[["record_id", "planning_status_normalized", "planning_maturity_score", "source_strength_score", "scale_score", "data_quality_penalty", "project_momentum_score", "project_momentum_label", "project_momentum_explanation"]].head())

Project Momentum Score label counts:


,project_momentum_label,num_records
0,HIGH,497
1,VERY_HIGH,386
2,MODERATE,53
3,LOW,2


,record_id,planning_status_normalized,planning_maturity_score,source_strength_score,scale_score,data_quality_penalty,project_momentum_score,project_momentum_label,project_momentum_explanation
0,UR_4001,PERMIT_APPROVED,90.0,100.0,75.0,0,92.0,VERY_HIGH,Advanced public status with strong planning-ma...
1,UR_4005,CONSTRUCTION,95.0,100.0,75.0,0,94.8,VERY_HIGH,Advanced public status with strong planning-ma...
2,UR_4006,PERMIT_APPROVED,90.0,100.0,75.0,0,92.0,VERY_HIGH,Advanced public status with strong planning-ma...
3,UR_4010,CONSTRUCTION,95.0,100.0,60.0,0,93.2,VERY_HIGH,Advanced public status with strong planning-ma...
4,UR_4011,CONSTRUCTION,95.0,100.0,90.0,0,96.2,VERY_HIGH,Advanced public status with strong planning-ma...


## 10 - Advanced Project Label for ML

Create a derived target from public planning status. This target is used only for a cautious similarity classifier, not a future prediction model.

In [10]:
ADVANCED_STATUSES = {"PLAN_APPROVED", "PERMIT_REQUESTED", "PERMIT_APPROVED", "CONSTRUCTION", "COMPLETED"}
NOT_ADVANCED_STATUSES = {"POLICY_AREA_ONLY", "DECLARED_COMPLEX", "PLAN_IN_PROGRESS", "PLAN_DEPOSITED", "UNKNOWN"}

def derive_advanced_label(status: Any) -> Any:
    if pd.isna(status):
        return pd.NA
    status = str(status)
    if status in ADVANCED_STATUSES:
        return 1
    if status in NOT_ADVANCED_STATUSES:
        return 0
    return pd.NA

scored_df["advanced_project_label"] = scored_df["planning_status_normalized"].apply(derive_advanced_label).astype("Int64")

print("Advanced public-status label counts:")
display(scored_df["advanced_project_label"].value_counts(dropna=False).rename_axis("advanced_project_label").reset_index(name="num_records"))

Advanced public-status label counts:


,advanced_project_label,num_records
0,1,552
1,0,386


## 11 - ML Feature Set

Build a leakage-aware feature set. Status fields and rule-based momentum fields are forbidden as ML features.

In [11]:
ORIGINAL_CATEGORICAL_FEATURES = ["city", "renewal_type", "confidence_level"]
ORIGINAL_BOOLEAN_FEATURES = [
    "has_plan_number",
    "has_mavat_url",
    "has_map_url",
    "has_existing_units",
    "has_proposed_units",
    "has_permits",
    "has_quality_issue",
    "declared_complex",
]
ORIGINAL_NUMERIC_FEATURES = [
    "existing_units",
    "additional_units",
    "proposed_units",
    "permits_total",
    "proposed_to_existing_ratio",
    "additional_to_existing_ratio",
    "declaration_year",
    "years_since_declaration",
    "validity_year",
    "data_confidence_score",
    "source_strength_score",
    "scale_score",
    "data_quality_penalty",
]

LEAKAGE_SAFE_CATEGORICAL_FEATURES = ["city", "renewal_type", "confidence_level"]
LEAKAGE_SAFE_BOOLEAN_FEATURES = [
    "has_plan_number",
    "has_mavat_url",
    "has_map_url",
    "has_existing_units",
    "has_proposed_units",
    "has_quality_issue",
    "declared_complex",
]
LEAKAGE_SAFE_NUMERIC_FEATURES = [
    "existing_units",
    "additional_units",
    "proposed_units",
    "proposed_to_existing_ratio",
    "additional_to_existing_ratio",
    "declaration_year",
    "years_since_declaration",
    "data_confidence_score",
    "source_strength_score",
    "scale_score",
    "data_quality_penalty",
]

LEAKAGE_SAFE_ML_FEATURES = (
    LEAKAGE_SAFE_CATEGORICAL_FEATURES
    + LEAKAGE_SAFE_BOOLEAN_FEATURES
    + LEAKAGE_SAFE_NUMERIC_FEATURES
)

FEATURE_SETS = {
    "original_feature_set": {
        "categorical": ORIGINAL_CATEGORICAL_FEATURES,
        "boolean": ORIGINAL_BOOLEAN_FEATURES,
        "numeric": ORIGINAL_NUMERIC_FEATURES,
        "notes": "Original cautious feature set retained for comparison; includes permit and validity-year public status proxies.",
    },
    "leakage_safe_feature_set": {
        "categorical": LEAKAGE_SAFE_CATEGORICAL_FEATURES,
        "boolean": LEAKAGE_SAFE_BOOLEAN_FEATURES,
        "numeric": LEAKAGE_SAFE_NUMERIC_FEATURES,
        "notes": "Dashboard feature set; status and strong permit/status proxy features excluded.",
    },
}

FORBIDDEN_ML_FEATURES = [
    "planning_status_raw",
    "planning_status_normalized",
    "status_normalization_reason",
    "in_execution",
    "planning_maturity_score",
    "project_momentum_score",
    "project_momentum_label",
    "lawyer_note",
    "dashboard_note",
    "advanced_project_label",
    "permits_total",
    "has_permits",
    "validity_year",
]

all_categorical_features = sorted({col for config in FEATURE_SETS.values() for col in config["categorical"]})
all_boolean_features = sorted({col for config in FEATURE_SETS.values() for col in config["boolean"]})
all_numeric_features = sorted({col for config in FEATURE_SETS.values() for col in config["numeric"]})

for col in all_categorical_features + all_boolean_features + all_numeric_features:
    if col not in scored_df.columns:
        scored_df[col] = pd.NA

for col in all_boolean_features:
    scored_df[col] = scored_df[col].fillna(False).astype(bool).astype(int)
for col in all_numeric_features:
    scored_df[col] = safe_numeric(scored_df[col])

for config in FEATURE_SETS.values():
    config["numeric_plus_boolean"] = config["numeric"] + config["boolean"]
    config["all"] = config["categorical"] + config["numeric_plus_boolean"]

ml_feature_columns = FEATURE_SETS["leakage_safe_feature_set"]["all"]
ml_df = scored_df.loc[
    scored_df["advanced_project_label"].notna(),
    sorted(set(sum((config["all"] for config in FEATURE_SETS.values()), []))) + ["advanced_project_label"],
].copy()

print(f"ML rows available: {len(ml_df):,}")
print("ML class counts:")
display(ml_df["advanced_project_label"].value_counts(dropna=False).rename_axis("advanced_project_label").reset_index(name="num_records"))
print("\nFeature sets:")
for feature_set_name, config in FEATURE_SETS.items():
    print(f"- {feature_set_name}: {len(config['all'])} features; {config['notes']}")

ML rows available: 938
ML class counts:


,advanced_project_label,num_records
0,1,552
1,0,386



Feature sets:
- original_feature_set: 24 features; Original cautious feature set retained for comparison; includes permit and validity-year public status proxies.
- leakage_safe_feature_set: 21 features; Dashboard feature set; status and strong permit/status proxy features excluded.


## 12 - Train ML Models

Train simple Logistic Regression and Random Forest similarity classifiers when scikit-learn and enough class variation are available.

In [12]:
ml_metrics_rows: List[Dict[str, Any]] = []
trained_models: Dict[Tuple[str, str], Any] = {}
ml_training_data_by_feature_set: Dict[str, Dict[str, Any]] = {}
chosen_model_name = "Not available"
chosen_model = None
chosen_feature_set_name = "leakage_safe_feature_set"
ml_training_note = "ML not attempted yet."


def make_preprocessor(config: Dict[str, Any]) -> ColumnTransformer:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, config["numeric_plus_boolean"]),
            ("categorical", categorical_transformer, config["categorical"]),
        ],
        remainder="drop",
    )


def make_model_specs() -> Dict[str, Any]:
    return {
        "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        "RandomForestClassifier": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced", min_samples_leaf=3),
    }


def make_pipeline(model_name: str, config: Dict[str, Any]) -> Pipeline:
    model_specs = make_model_specs()
    return Pipeline(steps=[("preprocess", make_preprocessor(config)), ("model", model_specs[model_name])])


def append_unavailable_metric(feature_set_name: str, note: str) -> None:
    ml_metrics_rows.append({
        "feature_set": feature_set_name,
        "model": "Not available",
        "accuracy": pd.NA,
        "precision": pd.NA,
        "recall": pd.NA,
        "f1": pd.NA,
        "roc_auc": pd.NA,
        "train_rows": 0,
        "test_rows": 0,
        "notes": note,
    })


if not SKLEARN_AVAILABLE:
    ml_training_note = f"sklearn unavailable: {SKLEARN_IMPORT_ERROR}"
    for feature_set_name in FEATURE_SETS:
        append_unavailable_metric(feature_set_name, ml_training_note)
else:
    for feature_set_name, config in FEATURE_SETS.items():
        X = ml_df[config["all"]].copy()
        y = ml_df["advanced_project_label"].astype(int)
        class_counts = y.value_counts()
        can_train = len(X) >= 20 and y.nunique() == 2 and class_counts.min() >= 2
        ml_training_data_by_feature_set[feature_set_name] = {
            "X": X,
            "y": y,
            "row_index": ml_df.index.copy(),
            "config": config,
            "can_train": can_train,
        }

        if not can_train:
            ml_training_note = "Not enough rows or class variation for ML similarity training."
            append_unavailable_metric(feature_set_name, ml_training_note)
            continue

        stratify_arg = y if class_counts.min() >= 2 else None
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.25,
            random_state=42,
            stratify=stratify_arg,
        )

        for model_name in make_model_specs():
            pipeline = make_pipeline(model_name, config)
            try:
                pipeline.fit(X_train, y_train)
                y_pred = pipeline.predict(X_test)
                y_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, "predict_proba") else None
                roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None and y_test.nunique() == 2 else pd.NA
                ml_metrics_rows.append({
                    "feature_set": feature_set_name,
                    "model": model_name,
                    "accuracy": accuracy_score(y_test, y_pred),
                    "precision": precision_score(y_test, y_pred, zero_division=0),
                    "recall": recall_score(y_test, y_pred, zero_division=0),
                    "f1": f1_score(y_test, y_pred, zero_division=0),
                    "roc_auc": roc_auc,
                    "train_rows": len(X_train),
                    "test_rows": len(X_test),
                    "notes": (
                        "Indicative ML similarity classifier only; target is derived from public planning status. "
                        + config["notes"]
                    ),
                })
                trained_models[(feature_set_name, model_name)] = pipeline
            except Exception as exc:
                ml_metrics_rows.append({
                    "feature_set": feature_set_name,
                    "model": model_name,
                    "accuracy": pd.NA,
                    "precision": pd.NA,
                    "recall": pd.NA,
                    "f1": pd.NA,
                    "roc_auc": pd.NA,
                    "train_rows": len(X_train) if "X_train" in locals() else 0,
                    "test_rows": len(X_test) if "X_test" in locals() else 0,
                    "notes": f"Training failed: {type(exc).__name__}: {exc}",
                })

ml_model_metrics_df = pd.DataFrame(ml_metrics_rows)
ml_model_metrics_df.to_csv(ML_MODEL_METRICS_PATH, index=False, encoding="utf-8-sig")

display(ml_model_metrics_df)

,feature_set,model,accuracy,precision,recall,f1,roc_auc,train_rows,test_rows,notes
0,original_feature_set,LogisticRegression,0.761702,0.910000,0.659420,0.764706,0.869416,703,235,Indicative ML similarity classifier only; targ...
1,original_feature_set,RandomForestClassifier,0.957447,0.963768,0.963768,0.963768,0.990811,703,235,Indicative ML similarity classifier only; targ...
2,leakage_safe_feature_set,LogisticRegression,0.659574,0.750000,0.630435,0.685039,0.774914,703,235,Indicative ML similarity classifier only; targ...
3,leakage_safe_feature_set,RandomForestClassifier,0.846809,0.892308,0.840580,0.865672,0.919095,703,235,Indicative ML similarity classifier only; targ...


## 13 - Choose ML Model

Choose the model by ROC AUC when available, otherwise F1. Prefer Logistic Regression when performance is similar.

In [13]:
DASHBOARD_ML_FEATURE_SET = "leakage_safe_feature_set"
ml_oof_note = "OOF scoring not attempted."
ml_model_persistence_note = "ML model persistence not attempted."


def choose_model_for_feature_set(feature_set_name: str) -> str:
    available_model_names = [model_name for fs_name, model_name in trained_models if fs_name == feature_set_name]
    if not available_model_names:
        return "Not available"
    metrics_for_choice = ml_model_metrics_df[
        (ml_model_metrics_df["feature_set"] == feature_set_name)
        & (ml_model_metrics_df["model"].isin(available_model_names))
    ].copy()
    metrics_for_choice["roc_auc_numeric"] = pd.to_numeric(metrics_for_choice["roc_auc"], errors="coerce")
    metrics_for_choice["f1_numeric"] = pd.to_numeric(metrics_for_choice["f1"], errors="coerce")

    if metrics_for_choice["roc_auc_numeric"].notna().any():
        best_auc = metrics_for_choice["roc_auc_numeric"].max()
        near_best = metrics_for_choice[metrics_for_choice["roc_auc_numeric"] >= best_auc - 0.01]
        return "LogisticRegression" if "LogisticRegression" in near_best["model"].values else near_best.sort_values("roc_auc_numeric", ascending=False).iloc[0]["model"]

    best_f1 = metrics_for_choice["f1_numeric"].max()
    near_best = metrics_for_choice[metrics_for_choice["f1_numeric"] >= best_f1 - 0.01]
    return "LogisticRegression" if "LogisticRegression" in near_best["model"].values else near_best.sort_values("f1_numeric", ascending=False).iloc[0]["model"]


chosen_feature_set_name = DASHBOARD_ML_FEATURE_SET
chosen_model_name = choose_model_for_feature_set(chosen_feature_set_name)
chosen_model = None
scored_df["ml_advancement_score_oof"] = pd.NA

if chosen_model_name != "Not available":
    training_bundle = ml_training_data_by_feature_set[chosen_feature_set_name]
    chosen_config = training_bundle["config"]
    X_all = training_bundle["X"]
    y_all = training_bundle["y"]
    row_index = training_bundle["row_index"]
    class_counts = y_all.value_counts()
    n_splits = int(min(5, class_counts.min())) if y_all.nunique() == 2 else 0

    try:
        if n_splits >= 2:
            oof_probabilities = pd.Series(np.nan, index=scored_df.index, dtype="float64")
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
            for train_idx, valid_idx in skf.split(X_all, y_all):
                fold_model = make_pipeline(chosen_model_name, chosen_config)
                fold_model.fit(X_all.iloc[train_idx], y_all.iloc[train_idx])
                fold_proba = fold_model.predict_proba(X_all.iloc[valid_idx])[:, 1]
                oof_probabilities.loc[row_index[valid_idx]] = fold_proba
            scored_df["ml_advancement_score_oof"] = np.round(oof_probabilities * 100, 1)
            scored_df["ml_advancement_score"] = scored_df["ml_advancement_score_oof"]
            ml_oof_note = f"Out-of-fold probabilities created with StratifiedKFold(n_splits={n_splits})."
        else:
            raise ValueError("Not enough class members for StratifiedKFold OOF scoring.")
    except Exception as exc:
        full_model_for_fallback = trained_models[(chosen_feature_set_name, chosen_model_name)]
        fallback_scores = full_model_for_fallback.predict_proba(scored_df[chosen_config["all"]].copy())[:, 1] * 100
        scored_df["ml_advancement_score"] = np.round(fallback_scores, 1)
        ml_oof_note = f"OOF scoring failed; fell back to full-model probabilities. {type(exc).__name__}: {exc}"

    chosen_model = make_pipeline(chosen_model_name, chosen_config)
    chosen_model.fit(X_all, y_all)

    ml_similarity_metadata = {
        "model_name": chosen_model_name,
        "feature_set_used": chosen_feature_set_name,
        "categorical_features": list(chosen_config["categorical"]),
        "numeric_features": list(chosen_config["numeric"]),
        "boolean_features": list(chosen_config["boolean"]),
        "numeric_plus_boolean_features": list(chosen_config["numeric_plus_boolean"]),
        "all_features": list(chosen_config["all"]),
        "target_name": "advanced_project_label",
        "score_column": "ml_advancement_score",
        "label_column": "ml_advancement_label",
        "label_thresholds": {
            "HIGH_SIMILARITY": "score >= 70",
            "MEDIUM_SIMILARITY": "40 <= score < 70",
            "LOW_SIMILARITY": "score < 40",
        },
        "warning": "Indicative ML similarity score only. Not a prediction and not legal/planning advice.",
        "notes": "Pipeline is fitted on the full leakage-safe ML training dataset for dashboard inference on uploaded public records.",
        "saved_at": datetime.now().isoformat(timespec="seconds"),
    }
    joblib.dump(chosen_model, ML_SIMILARITY_MODEL_PATH)
    ML_SIMILARITY_METADATA_PATH.write_text(
        json.dumps(ml_similarity_metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    ml_model_persistence_note = f"Saved fitted ML pipeline to {ML_SIMILARITY_MODEL_PATH.relative_to(PROJECT_ROOT)} and metadata to {ML_SIMILARITY_METADATA_PATH.relative_to(PROJECT_ROOT)}."

    scored_df["ml_advancement_label"] = scored_df["ml_advancement_score"].apply(ml_similarity_label)
    scored_df["ml_model_used"] = chosen_model_name
else:
    scored_df["ml_advancement_score"] = pd.NA
    scored_df["ml_advancement_label"] = "ML_NOT_AVAILABLE"
    scored_df["ml_model_used"] = "Not available"
    chosen_feature_set_name = "leakage_safe_feature_set"
    ml_model_persistence_note = "ML model unavailable; no fitted pipeline saved."

scored_df["ml_feature_set_used"] = chosen_feature_set_name
scored_df["ml_score_warning"] = (
    "Indicative ML similarity score only. Not a prediction and not legal/planning advice. "
    "Leakage-safe feature set used; status and strong permit/status proxy features excluded."
)

print(f"Chosen ML feature set: {chosen_feature_set_name}")
print(f"Chosen ML model: {chosen_model_name}")
print(ml_oof_note)
print(ml_model_persistence_note)
display(scored_df[["record_id", "ml_advancement_score", "ml_advancement_score_oof", "ml_advancement_label", "ml_model_used", "ml_feature_set_used"]].head())

Chosen ML feature set: leakage_safe_feature_set
Chosen ML model: RandomForestClassifier
Out-of-fold probabilities created with StratifiedKFold(n_splits=5).
Saved fitted ML pipeline to models\ml_similarity_pipeline.joblib and metadata to models\ml_similarity_metadata.json.


,record_id,ml_advancement_score,ml_advancement_score_oof,ml_advancement_label,ml_model_used,ml_feature_set_used
0,UR_4001,90.5,90.5,HIGH_SIMILARITY,RandomForestClassifier,leakage_safe_feature_set
1,UR_4005,85.6,85.6,HIGH_SIMILARITY,RandomForestClassifier,leakage_safe_feature_set
2,UR_4006,91.3,91.3,HIGH_SIMILARITY,RandomForestClassifier,leakage_safe_feature_set
3,UR_4010,84.2,84.2,HIGH_SIMILARITY,RandomForestClassifier,leakage_safe_feature_set
4,UR_4011,88.1,88.1,HIGH_SIMILARITY,RandomForestClassifier,leakage_safe_feature_set


## 14 - Feature Importance

Extract available model importance values for interpretability.

In [14]:
feature_importance_rows: List[Dict[str, Any]] = []


def get_feature_names_from_pipeline(pipeline: Pipeline, fallback_features: List[str]) -> List[str]:
    try:
        preprocessor = pipeline.named_steps["preprocess"]
        return list(preprocessor.get_feature_names_out())
    except Exception:
        return fallback_features


if chosen_model is not None:
    chosen_config = FEATURE_SETS[chosen_feature_set_name]
    feature_names = get_feature_names_from_pipeline(chosen_model, chosen_config["all"])
    model = chosen_model.named_steps["model"]
    if chosen_model_name == "LogisticRegression" and hasattr(model, "coef_"):
        importances = np.abs(model.coef_[0])
        for feature, importance in zip(feature_names, importances):
            feature_importance_rows.append({
                "feature_set": chosen_feature_set_name,
                "model": chosen_model_name,
                "feature": feature,
                "importance": float(importance),
                "importance_type": "absolute_coefficient",
            })
    elif chosen_model_name == "RandomForestClassifier" and hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        for feature, importance in zip(feature_names, importances):
            feature_importance_rows.append({
                "feature_set": chosen_feature_set_name,
                "model": chosen_model_name,
                "feature": feature,
                "importance": float(importance),
                "importance_type": "feature_importance",
            })
else:
    feature_importance_rows.append({
        "feature_set": "leakage_safe_feature_set",
        "model": "Not available",
        "feature": "Not available",
        "importance": pd.NA,
        "importance_type": "ML unavailable or not enough class variation",
    })

ml_feature_importance_df = pd.DataFrame(feature_importance_rows)
if not ml_feature_importance_df.empty and "importance" in ml_feature_importance_df.columns:
    ml_feature_importance_df = ml_feature_importance_df.sort_values("importance", ascending=False, na_position="last")
ml_feature_importance_df.to_csv(ML_FEATURE_IMPORTANCE_PATH, index=False, encoding="utf-8-sig")

display(ml_feature_importance_df.head(30))

,feature_set,model,feature,importance,importance_type
5,leakage_safe_feature_set,RandomForestClassifier,numeric__declaration_year,0.228910,feature_importance
6,leakage_safe_feature_set,RandomForestClassifier,numeric__years_since_declaration,0.224428,feature_importance
95,leakage_safe_feature_set,RandomForestClassifier,categorical__renewal_type_מיסוי,0.069226,feature_importance
1,leakage_safe_feature_set,RandomForestClassifier,numeric__additional_units,0.052196,feature_importance
0,leakage_safe_feature_set,RandomForestClassifier,numeric__existing_units,0.050637,feature_importance
2,leakage_safe_feature_set,RandomForestClassifier,numeric__proposed_units,0.045523,feature_importance
3,leakage_safe_feature_set,RandomForestClassifier,numeric__proposed_to_existing_ratio,0.043266,feature_importance
94,leakage_safe_feature_set,RandomForestClassifier,categorical__renewal_type_טרם הוכרז,0.043023,feature_importance
4,leakage_safe_feature_set,RandomForestClassifier,numeric__additional_to_existing_ratio,0.042271,feature_importance
96,leakage_safe_feature_set,RandomForestClassifier,categorical__renewal_type_רשויות,0.031887,feature_importance


## 15 - Reports

Create scoring and ML metric reports for auditability.

In [15]:
def count_label(label: str) -> int:
    return int((scored_df["project_momentum_label"] == label).sum())

scoring_report_rows = [
    ("input_rows", len(standardized_df), "Rows read from Notebook 02 standardized dataset."),
    ("output_rows", len(scored_df), "Rows written to scored dataset."),
    ("number_of_cities", int(scored_df["city"].nunique(dropna=True)), "Distinct cities."),
    ("advanced_project_label_0_count", int((scored_df["advanced_project_label"] == 0).sum()), "Rows with non-advanced public status label."),
    ("advanced_project_label_1_count", int((scored_df["advanced_project_label"] == 1).sum()), "Rows with advanced public status label."),
    ("mean_planning_maturity_score", float(scored_df["planning_maturity_score"].mean()), "Average public planning maturity score."),
    ("mean_source_strength_score", float(scored_df["source_strength_score"].mean()), "Average public-source strength score."),
    ("mean_scale_score", float(scored_df["scale_score"].mean()), "Average cautious project scale score."),
    ("mean_data_confidence_score", float(scored_df["data_confidence_score"].mean()), "Average Notebook 02 data confidence score."),
    ("mean_project_momentum_score", float(scored_df["project_momentum_score"].mean()), "Average Project Momentum Score."),
    ("very_high_momentum_records", count_label("VERY_HIGH"), "Rows with VERY_HIGH Project Momentum Score."),
    ("high_momentum_records", count_label("HIGH"), "Rows with HIGH Project Momentum Score."),
    ("moderate_momentum_records", count_label("MODERATE"), "Rows with MODERATE Project Momentum Score."),
    ("low_momentum_records", count_label("LOW"), "Rows with LOW Project Momentum Score."),
    ("records_with_quality_issues", int(scored_df["has_quality_issue"].sum()), "Rows with non-OK data_quality_flag."),
    ("records_with_ml_score", int(scored_df["ml_advancement_score"].notna().sum()), "Rows with available ML Similarity Score."),
    ("records_with_oof_ml_score", int(scored_df["ml_advancement_score_oof"].notna().sum()), "Rows with out-of-fold ML Similarity Score."),
    ("ml_model_used", chosen_model_name, "Chosen ML similarity model, if available."),
    ("ml_feature_set_used", chosen_feature_set_name, "Dashboard ML feature set."),
    ("ml_oof_note", ml_oof_note, "OOF scoring status."),
    ("ml_model_persistence_note", ml_model_persistence_note, "Saved dashboard inference model status."),
    ("ml_similarity_pipeline_path", str(ML_SIMILARITY_MODEL_PATH.relative_to(PROJECT_ROOT)), "Saved fitted ML pipeline for dashboard inference."),
    ("ml_similarity_metadata_path", str(ML_SIMILARITY_METADATA_PATH.relative_to(PROJECT_ROOT)), "Saved ML feature metadata for dashboard inference."),
]

scoring_report_df = pd.DataFrame(scoring_report_rows, columns=["metric", "value", "notes"])
scoring_report_df.to_csv(SCORING_REPORT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved scoring report: {SCORING_REPORT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved ML metrics: {ML_MODEL_METRICS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved ML feature importance: {ML_FEATURE_IMPORTANCE_PATH.relative_to(PROJECT_ROOT)}")
display(scoring_report_df)

Saved scoring report: data\metadata\scoring_report.csv
Saved ML metrics: data\metadata\ml_model_metrics.csv
Saved ML feature importance: data\metadata\ml_feature_importance.csv


,metric,value,notes
0,input_rows,938,Rows read from Notebook 02 standardized dataset.
1,output_rows,938,Rows written to scored dataset.
2,number_of_cities,76,Distinct cities.
3,advanced_project_label_0_count,386,Rows with non-advanced public status label.
4,advanced_project_label_1_count,552,Rows with advanced public status label.
5,mean_planning_maturity_score,62.739872,Average public planning maturity score.
6,mean_source_strength_score,96.934968,Average public-source strength score.
7,mean_scale_score,71.988273,Average cautious project scale score.
8,mean_data_confidence_score,98.720682,Average Notebook 02 data confidence score.
9,mean_project_momentum_score,75.849254,Average Project Momentum Score.


## 16 - Summary Tables

Create city-level and status-level scored summaries for dashboard use.

In [16]:
boolean_count_cols = [
    "has_plan_number",
    "has_mavat_url",
    "has_quality_issue",
    "has_map_url",
    "has_permits",
    "has_existing_units",
    "has_proposed_units",
]
for col in boolean_count_cols:
    if col not in scored_df.columns:
        scored_df[col] = False
    scored_df[col] = scored_df[col].fillna(False).astype(bool)

city_group = scored_df.groupby("city", dropna=False)
city_scored_summary_df = city_group.agg(
    num_records=("record_id", "size"),
    avg_project_momentum_score=("project_momentum_score", "mean"),
    median_project_momentum_score=("project_momentum_score", "median"),
    avg_ml_advancement_score=("ml_advancement_score", "mean"),
    num_with_plan_number=("has_plan_number", "sum"),
    num_with_mavat_url=("has_mavat_url", "sum"),
    num_with_quality_issues=("has_quality_issue", "sum"),
    total_existing_units=("existing_units", "sum"),
    total_proposed_units=("proposed_units", "sum"),
).reset_index()

integer_count_cols = ["num_records", "num_with_plan_number", "num_with_mavat_url", "num_with_quality_issues"]
for col in integer_count_cols:
    city_scored_summary_df[col] = city_scored_summary_df[col].fillna(0).astype(int)

city_scored_summary_df["num_very_high_momentum"] = city_group["project_momentum_label"].apply(lambda s: int((s == "VERY_HIGH").sum())).values
city_scored_summary_df["num_high_momentum"] = city_group["project_momentum_label"].apply(lambda s: int((s == "HIGH").sum())).values
city_scored_summary_df["num_moderate_momentum"] = city_group["project_momentum_label"].apply(lambda s: int((s == "MODERATE").sum())).values
city_scored_summary_df["num_low_momentum"] = city_group["project_momentum_label"].apply(lambda s: int((s == "LOW").sum())).values
city_scored_summary_df["num_advanced_public_status"] = city_group["advanced_project_label"].apply(lambda s: int((s == 1).sum())).values
city_scored_summary_df["num_plan_approved"] = city_group["planning_status_normalized"].apply(lambda s: int((s == "PLAN_APPROVED").sum())).values
city_scored_summary_df["num_permit_approved"] = city_group["planning_status_normalized"].apply(lambda s: int((s == "PERMIT_APPROVED").sum())).values
city_scored_summary_df["num_construction"] = city_group["planning_status_normalized"].apply(lambda s: int((s == "CONSTRUCTION").sum())).values
city_scored_summary_df = city_scored_summary_df[
    [
        "city",
        "num_records",
        "avg_project_momentum_score",
        "median_project_momentum_score",
        "num_very_high_momentum",
        "num_high_momentum",
        "num_moderate_momentum",
        "num_low_momentum",
        "avg_ml_advancement_score",
        "num_advanced_public_status",
        "num_plan_approved",
        "num_permit_approved",
        "num_construction",
        "num_with_plan_number",
        "num_with_mavat_url",
        "num_with_quality_issues",
        "total_existing_units",
        "total_proposed_units",
    ]
]
city_scored_summary_df.to_csv(CITY_SCORED_SUMMARY_PATH, index=False, encoding="utf-8-sig")

status_scored_summary_df = (
    scored_df.groupby("planning_status_normalized", dropna=False)
    .agg(
        num_records=("record_id", "size"),
        avg_project_momentum_score=("project_momentum_score", "mean"),
        avg_ml_advancement_score=("ml_advancement_score", "mean"),
        avg_data_confidence_score=("data_confidence_score", "mean"),
        avg_source_strength_score=("source_strength_score", "mean"),
    )
    .reset_index()
)
status_scored_summary_df["share_of_records"] = status_scored_summary_df["num_records"] / len(scored_df) if len(scored_df) else 0
status_scored_summary_df = status_scored_summary_df[
    [
        "planning_status_normalized",
        "num_records",
        "share_of_records",
        "avg_project_momentum_score",
        "avg_ml_advancement_score",
        "avg_data_confidence_score",
        "avg_source_strength_score",
    ]
]
status_scored_summary_df.to_csv(STATUS_SCORED_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print(f"Saved city scored summary: {CITY_SCORED_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved status scored summary: {STATUS_SCORED_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")
display(city_scored_summary_df.head())
display(status_scored_summary_df)

Saved city scored summary: data\processed\urban_renewal_city_scored_summary.csv
Saved status scored summary: data\processed\urban_renewal_scored_status_summary.csv


,city,num_records,avg_project_momentum_score,median_project_momentum_score,num_very_high_momentum,num_high_momentum,num_moderate_momentum,num_low_momentum,avg_ml_advancement_score,num_advanced_public_status,num_plan_approved,num_permit_approved,num_construction,num_with_plan_number,num_with_mavat_url,num_with_quality_issues,total_existing_units,total_proposed_units
0,אבן יהודה,2,64.500000,64.5,0,2,0,0,33.750000,0,0,0,0,2,2,0,216.0,684.0
1,אור יהודה,9,81.488889,81.0,6,3,0,0,62.911111,7,4,1,2,9,8,0,3057.0,10991.0
2,אור עקיבא,1,83.500000,83.5,1,0,0,0,79.300000,1,1,0,0,1,1,0,350.0,1750.0
3,אזור,5,77.660000,82.5,3,2,0,0,49.700000,3,2,0,1,5,5,0,1405.0,4406.0
4,אילת,6,64.250000,64.5,0,6,0,0,32.166667,0,0,0,0,6,6,0,810.0,2340.0


,planning_status_normalized,num_records,share_of_records,avg_project_momentum_score,avg_ml_advancement_score,avg_data_confidence_score,avg_source_strength_score
0,CONSTRUCTION,54,0.057569,95.344444,76.816667,99.814815,99.537037
1,PERMIT_APPROVED,173,0.184435,90.658960,80.950289,99.479769,98.699422
2,PLAN_APPROVED,325,0.346482,80.192308,61.495385,99.446154,98.615385
3,PLAN_IN_PROGRESS,385,0.410448,62.990909,33.361299,97.714286,94.415584
4,UNKNOWN,1,0.001066,0.000000,36.000000,60.000000,75.000000


## 17 - Dashboard Note

Create concise dashboard notes that keep the public-data nature and manual-verification need clear.

In [17]:
scored_df["dashboard_note"] = scored_df.apply(create_dashboard_note, axis=1)

display(scored_df[["record_id", "project_momentum_label", "data_quality_flag", "dashboard_note"]].head())

,record_id,project_momentum_label,data_quality_flag,dashboard_note
0,UR_4001,VERY_HIGH,OK,Public data indicates an advanced or highly ma...
1,UR_4005,VERY_HIGH,OK,Public data indicates an advanced or highly ma...
2,UR_4006,VERY_HIGH,OK,Public data indicates an advanced or highly ma...
3,UR_4010,VERY_HIGH,OK,Public data indicates an advanced or highly ma...
4,UR_4011,VERY_HIGH,OK,Public data indicates an advanced or highly ma...


## 18 - Final Save

Save the full scored dataset. No rows are dropped.

In [18]:
SCORED_DATA_DICTIONARY_ENTRIES = [
    ("has_plan_number", "Whether a plan number is present.", "plan_number", "boolean", "Source traceability feature."),
    ("has_mavat_url", "Whether a Mavat/source URL is present.", "mavat_url", "boolean", "Source traceability feature."),
    ("has_map_url", "Whether a map URL is present.", "map_url", "boolean", "Source traceability feature."),
    ("has_existing_units", "Whether positive existing units are present.", "existing_units", "boolean", "Scale feature."),
    ("has_proposed_units", "Whether positive proposed units are present.", "proposed_units", "boolean", "Scale feature."),
    ("has_permits", "Whether permits_total is positive.", "permits_total", "boolean", "Source feature, not legal finding."),
    ("has_quality_issue", "Whether data_quality_flag is not OK.", "data_quality_flag", "boolean", "Used for penalty."),
    ("declaration_year", "Year parsed from declaration_date.", "declaration_date", "integer", "Missing if date missing."),
    ("years_since_declaration", "Current year minus declaration year.", "declaration_year", "integer", "Indicative age only."),
    ("proposed_to_existing_ratio", "Proposed units divided by existing units.", "proposed_units/existing_units", "float", "Missing for invalid/negative/anomalous values."),
    ("additional_to_existing_ratio", "Additional units divided by existing units.", "additional_units/existing_units", "float", "Missing for invalid/negative/anomalous values."),
    ("planning_maturity_score", "Rule-based public planning maturity score.", "planning_status_normalized", "0-100", "Primary maturity component."),
    ("source_strength_score", "Traceable public-source support score.", "source fields", "0-100", "Higher when links and IDs exist."),
    ("scale_score", "Cautious project scale signal.", "unit fields", "0-100", "Not progress or legal value."),
    ("data_quality_penalty", "Penalty from data_quality_flag.", "data_quality_flag", "0-50", "Subtracted from momentum."),
    ("project_momentum_score", "Main explainable Project Momentum Score.", "derived", "0-100", "Indicative public-data maturity signal."),
    ("project_momentum_label", "Label for Project Momentum Score.", "project_momentum_score", "category", "VERY_HIGH/HIGH/MODERATE/LOW."),
    ("project_momentum_explanation", "Human-readable explanation of score drivers.", "derived", "string", "Cautious language only."),
    ("advanced_project_label", "Derived ML target from public status.", "planning_status_normalized", "0/1", "Not a future outcome."),
    ("ml_advancement_score_oof", "Out-of-fold ML Similarity Score before dashboard scaling/fallback.", "ML model", "0-100", "Preferred over in-sample scoring when available."),
    ("ml_advancement_score", "ML Similarity Score to publicly advanced projects.", "ML model", "0-100", "Leakage-safe feature set; not a prediction."),
    ("ml_advancement_label", "Label for ML Similarity Score.", "ml_advancement_score", "category", "Similarity only."),
    ("ml_model_used", "Chosen ML model name.", "model selection", "string", "Not available if sklearn unavailable."),
    ("ml_feature_set_used", "ML feature set used for dashboard score.", "model selection", "string", "Uses leakage_safe_feature_set when available."),
    ("ml_score_warning", "Warning text for ML score.", "constant", "string", "Display in dashboard."),
    ("dashboard_note", "Short dashboard-facing note.", "derived", "string", "Manual verification reminder."),
]

scored_data_dictionary_df = pd.DataFrame(
    SCORED_DATA_DICTIONARY_ENTRIES,
    columns=["column_name", "description", "source_column", "data_type", "notes"],
)
scored_data_dictionary_df.to_csv(SCORED_DATA_DICTIONARY_PATH, index=False, encoding="utf-8-sig")

scored_df.to_csv(OUTPUT_SCORED_PATH, index=False, encoding="utf-8-sig")

print(f"Saved scored dataset: {OUTPUT_SCORED_PATH.relative_to(PROJECT_ROOT)}")
print(f"Shape: {scored_df.shape[0]:,} rows x {scored_df.shape[1]:,} columns")
print("\nProject Momentum label counts:")
display(scored_df["project_momentum_label"].value_counts(dropna=False).rename_axis("project_momentum_label").reset_index(name="num_records"))
print("\nML advancement label counts:")
display(scored_df["ml_advancement_label"].value_counts(dropna=False).rename_axis("ml_advancement_label").reset_index(name="num_records"))

selected_columns = [
    "record_id",
    "city",
    "complex_name",
    "planning_status_normalized",
    "planning_maturity_score",
    "project_momentum_score",
    "project_momentum_label",
    "ml_advancement_score",
    "ml_advancement_label",
    "data_quality_flag",
]
display(scored_df[selected_columns].head())

Saved scored dataset: data\processed\urban_renewal_scored.csv
Shape: 938 rows x 61 columns

Project Momentum label counts:


,project_momentum_label,num_records
0,HIGH,497
1,VERY_HIGH,386
2,MODERATE,53
3,LOW,2



ML advancement label counts:


,ml_advancement_label,num_records
0,HIGH_SIMILARITY,372
1,LOW_SIMILARITY,355
2,MEDIUM_SIMILARITY,211


,record_id,city,complex_name,planning_status_normalized,planning_maturity_score,project_momentum_score,project_momentum_label,ml_advancement_score,ml_advancement_label,data_quality_flag
0,UR_4001,גבעתים,ערבי נחל,PERMIT_APPROVED,90.0,92.0,VERY_HIGH,90.5,HIGH_SIMILARITY,OK
1,UR_4005,קרית אונו,ישעיהו,CONSTRUCTION,95.0,94.8,VERY_HIGH,85.6,HIGH_SIMILARITY,OK
2,UR_4006,קרית אונו,שאול המלך,PERMIT_APPROVED,90.0,92.0,VERY_HIGH,91.3,HIGH_SIMILARITY,OK
3,UR_4010,ראשון לציון,רמת אליהו (פוזננסקי),CONSTRUCTION,95.0,93.2,VERY_HIGH,84.2,HIGH_SIMILARITY,OK
4,UR_4011,ראשון לציון,סלע,CONSTRUCTION,95.0,96.2,VERY_HIGH,88.1,HIGH_SIMILARITY,OK


## 19 - Final Summary

Confirm created outputs and downstream readiness.

In [19]:
print("Notebook 03 completed.")
print("\nCreated:")
for path in [
    OUTPUT_SCORED_PATH,
    CITY_SCORED_SUMMARY_PATH,
    STATUS_SCORED_SUMMARY_PATH,
    SCORING_REPORT_PATH,
    ML_MODEL_METRICS_PATH,
    ML_FEATURE_IMPORTANCE_PATH,
    SCORED_DATA_DICTIONARY_PATH,
    ML_SIMILARITY_MODEL_PATH,
    ML_SIMILARITY_METADATA_PATH,
]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

print("\nScored dataset is ready for Notebook 04 Streamlit dashboard.")

Notebook 03 completed.

Created:
- data\processed\urban_renewal_scored.csv
- data\processed\urban_renewal_city_scored_summary.csv
- data\processed\urban_renewal_scored_status_summary.csv
- data\metadata\scoring_report.csv
- data\metadata\ml_model_metrics.csv
- data\metadata\ml_feature_importance.csv
- data\metadata\urban_renewal_scored_data_dictionary.csv
- models\ml_similarity_pipeline.joblib
- models\ml_similarity_metadata.json

Scored dataset is ready for Notebook 04 Streamlit dashboard.
